In [2]:
import torch
from torch.utils.data import Dataset
import pickle

# AA to index (21개: 20 AA + gap)
AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20  # gap
}

def aa_sequence_to_indices(seq):
    return [AA_TO_INDEX.get(aa, 20) for aa in seq]

class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61):

        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df = df
        self.max_depth = max_depth
        self.win_size = win_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["UniProtID"]
        mut_pos = int(row["MutPos"]) - 1  # 1-based → 0-based 변환
        label = int(row["Label"])
        mut = row["Mut"].upper()

        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]  # 보통 첫 줄이 ref seq

        # mut seq 생성 (ref seq 복사 후 변이만 반영)
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq):
            mut_seq[mut_pos] = mut  # 변이 반영

        seqs_to_use = [mut_seq, list(query_seq)]  # mut seq + ref seq
        if len(msa_seqs) > 1:
            seqs_to_use += [list(seq) for seq in msa_seqs[1:self.max_depth-2]]

        half_win = self.win_size // 2
        centered_msa = []
        for seq in seqs_to_use:
            window = []
            for i in range(self.win_size):
                seq_idx = mut_pos - half_win + i
                if 0 <= seq_idx < len(seq):
                    aa = seq[seq_idx]
                else:
                    aa = '-'
                window.append(AA_TO_INDEX.get(aa, 20))
            centered_msa.append(window)
        # Padding (depth)
        if len(centered_msa) < self.max_depth:
            pad_len = self.max_depth - len(centered_msa)
            centered_msa += [[20] * self.win_size] * pad_len

        msa_tensor = torch.tensor(centered_msa[:self.max_depth]).long()  # [D, L]
        msa_tensor = msa_tensor.transpose(0, 1)  # [L, D]

        return {
            "msa": msa_tensor,  # [L, D], D=mut seq+ref seq+MSA homologs+pad
            "label": torch.tensor(label).long()
        }

In [3]:
# Copyright (c) 2024, Tri Dao, Albert Gu.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat

try:
    from causal_conv1d import causal_conv1d_fn
except ImportError:
    causal_conv1d_fn = None

try:
    from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated, LayerNorm
except ImportError:
    RMSNormGated, LayerNorm = None, None

from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined
from mamba_ssm.ops.triton.ssd_combined import mamba_split_conv1d_scan_combined


class Mamba2Simple(nn.Module):
    def __init__(
        self,
        d_model,
        d_state=64,
        d_conv=4,
        conv_init=None,
        expand=2,
        headdim=128,
        ngroups=1,
        A_init_range=(1, 16),
        dt_min=0.001,
        dt_max=0.1,
        dt_init_floor=1e-4,
        dt_limit=(0.0, float("inf")),
        learnable_init_states=False,
        activation="swish",
        bias=False,
        conv_bias=True,
        # Fused kernel and sharding options
        chunk_size=256,
        use_mem_eff_path=True,
        layer_idx=None,  # Absorb kwarg for general module
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.conv_init = conv_init
        self.expand = expand
        self.d_inner = self.expand * self.d_model
        self.headdim = headdim
        self.ngroups = ngroups
        assert self.d_inner % self.headdim == 0
        self.nheads = self.d_inner // self.headdim
        self.dt_limit = dt_limit
        self.learnable_init_states = learnable_init_states
        self.activation = activation
        self.chunk_size = chunk_size
        self.use_mem_eff_path = use_mem_eff_path
        self.layer_idx = layer_idx

        # Order: [z, x, B, C, dt]
        d_in_proj = 2 * self.d_inner + 2 * self.ngroups * self.d_state + self.nheads
        self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=bias, **factory_kwargs)

        conv_dim = self.d_inner + 2 * self.ngroups * self.d_state
        self.conv1d = nn.Conv1d(
            in_channels=conv_dim,
            out_channels=conv_dim,
            bias=conv_bias,
            kernel_size=d_conv,
            groups=conv_dim,
            padding=d_conv - 1,
            **factory_kwargs,
        )
        if self.conv_init is not None:
            nn.init.uniform_(self.conv1d.weight, -self.conv_init, self.conv_init)
        # self.conv1d.weight._no_weight_decay = True

        if self.learnable_init_states:
            self.init_states = nn.Parameter(torch.zeros(self.nheads, self.headdim, self.d_state, **factory_kwargs))
            self.init_states._no_weight_decay = True

        self.act = nn.SiLU()

        # Initialize log dt bias
        dt = torch.exp(
            torch.rand(self.nheads, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        dt = torch.clamp(dt, min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        # Just to be explicit. Without this we already don't put wd on dt_bias because of the check
        # name.endswith("bias") in param_grouping.py
        self.dt_bias._no_weight_decay = True

        # A parameter
        assert A_init_range[0] > 0 and A_init_range[1] >= A_init_range[0]
        A = torch.empty(self.nheads, dtype=torch.float32, device=device).uniform_(*A_init_range)
        A_log = torch.log(A).to(dtype=dtype)
        self.A_log = nn.Parameter(A_log)
        # self.register_buffer("A_log", torch.zeros(self.nheads, dtype=torch.float32, device=device), persistent=True)
        self.A_log._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.nheads, device=device))
        self.D._no_weight_decay = True

        # Extra normalization layer right before output projection
        assert RMSNormGated is not None
        self.norm = RMSNormGated(self.d_inner, eps=1e-5, norm_before_gate=False, **factory_kwargs)

        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)

    def forward(self, u, seq_idx=None):
        """
        u: (B, L, D)
        Returns: same shape as u
        """
        batch, seqlen, dim = u.shape

        zxbcdt = self.in_proj(u)  # (B, L, d_in_proj)
        A = -torch.exp(self.A_log)  # (nheads) or (d_inner, d_state)
        initial_states=repeat(self.init_states, "... -> b ...", b=batch) if self.learnable_init_states else None
        dt_limit_kwargs = {} if self.dt_limit == (0.0, float("inf")) else dict(dt_limit=self.dt_limit)

        if self.use_mem_eff_path:
            # Fully fused path
            out = mamba_split_conv1d_scan_combined(
                zxbcdt,
                rearrange(self.conv1d.weight, "d 1 w -> d w"),
                self.conv1d.bias,
                self.dt_bias,
                A,
                D=self.D,
                chunk_size=self.chunk_size,
                seq_idx=seq_idx,
                activation=self.activation,
                rmsnorm_weight=self.norm.weight,
                rmsnorm_eps=self.norm.eps,
                outproj_weight=self.out_proj.weight,
                outproj_bias=self.out_proj.bias,
                headdim=self.headdim,
                ngroups=self.ngroups,
                norm_before_gate=False,
                initial_states=initial_states,
                **dt_limit_kwargs,
            )
        else:
            z, xBC, dt = torch.split(
                zxbcdt, [self.d_inner, self.d_inner + 2 * self.ngroups * self.d_state, self.nheads], dim=-1
            )
            dt = F.softplus(dt + self.dt_bias)  # (B, L, nheads)
            assert self.activation in ["silu", "swish"]

            # 1D Convolution
            if causal_conv1d_fn is None or self.activation not in ["silu", "swish"]:
                xBC = self.act(
                    self.conv1d(xBC.transpose(1, 2)).transpose(1, 2)
                )  # (B, L, self.d_inner + 2 * ngroups * d_state)
                xBC = xBC[:, :seqlen, :]
            else:
                xBC = causal_conv1d_fn(
                    x=xBC.transpose(1, 2),
                    weight=rearrange(self.conv1d.weight, "d 1 w -> d w"),
                    bias=self.conv1d.bias,
                    activation=self.activation,
                ).transpose(1, 2)

            # Split into 3 main branches: X, B, C
            # These correspond to V, K, Q respectively in the SSM/attention duality
            x, B, C = torch.split(xBC, [self.d_inner, self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)
            y = mamba_chunk_scan_combined(
                rearrange(x, "b l (h p) -> b l h p", p=self.headdim),
                dt,
                A,
                rearrange(B, "b l (g n) -> b l g n", g=self.ngroups),
                rearrange(C, "b l (g n) -> b l g n", g=self.ngroups),
                chunk_size=self.chunk_size,
                D=self.D,
                z=None,
                seq_idx=seq_idx,
                initial_states=initial_states,
                **dt_limit_kwargs,
            )
            y = rearrange(y, "b l h p -> b l (h p)")

            # Multiply "gate" branch and apply extra normalization layer
            y = self.norm(y, z)
            out = self.out_proj(y)
        return out

/home/kunny/miniconda3/envs/mamba_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba
from mamba_ssm import Mamba2


# --- Input Embedding ---
class MSAInputEmbedding(nn.Module):
    def __init__(self, vocab_size=21, dim=256):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=dim)

    def forward(self, x):  # x: (B, L, D)
        return self.embedding(x)  # → (B, L, D, C)

# --- MambaRMSNorm ---
class MambaRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) / (x.shape[-1] ** 0.5)
        return self.weight * x / (norm + self.eps)

# --- Cross-Axial Mamba Block ---
class CrossAxialMambaMSA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm_L = MambaRMSNorm(dim)
        self.norm_D = MambaRMSNorm(dim)
        self.mamba_L = Mamba(d_model=dim, expand=1)
        self.mamba_D = Mamba(d_model=dim, expand=1)

    def forward(self, x):  # x: (B, L, D, C)
        B, L, D, C = x.shape

        # D-axis
        x_d = self.norm_D(x).view(B * L, D, C)
        d_out = self.mamba_D(x_d).view(B, L, D, C)

        # L-axis
        x_l = self.norm_L(x).permute(0, 2, 1, 3).contiguous().view(B * D, L, C)
        l_out = self.mamba_L(x_l).view(B, D, L, C).permute(0, 2, 1, 3)

        return x + d_out + l_out


# --- Cross-Axial Mamba Block ---
# class CrossAxialMambaMSA(nn.Module):
#     def __init__(self, dim):
#         super().__init__()
#         self.norm_L = MambaRMSNorm(dim)
#         self.norm_D = MambaRMSNorm(dim)
#         self.mamba_L = Mamba2(
#             d_model=dim,
#             expand=1, 
#             d_state=64,      
#             d_conv=4,         
#             rmsnorm=False,     
#         )

#         self.mamba_D = Mamba2(
#             d_model=dim,
#             expand=1,
#             d_state=64,
#             d_conv=4,
#             rmsnorm=False,
#         )

#     def forward(self, x):  # x: (B, L, D, C)
#         B, L, D, C = x.shape

#         # D-axis
#         x_d = self.norm_D(x).view(B * L, D, C)
#         d_out = self.mamba_D(x_d).view(B, L, D, C)

#         # L-axis
#         x_l = self.norm_L(x).permute(0, 2, 1, 3).contiguous().view(B * D, L, C)
#         l_out = self.mamba_L(x_l).view(B, D, L, C).permute(0, 2, 1, 3)

#         return x + d_out + l_out

# # --- Encoder ---
class MSAEncoder(nn.Module):
    def __init__(self, num_layers=8, dim=256):
        super().__init__()
        self.embeddings = MSAInputEmbedding(dim=dim)
        self.blocks = nn.ModuleList([
            CrossAxialMambaMSA(dim) for _ in range(num_layers)
        ])
        self.norm_f = MambaRMSNorm(dim)

    def forward(self, x):  # x: (B, L, D)
        x = self.embeddings(x)  # → (B, L, D, C)
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x)  # (B, L, D, C)

# --- Classifier Head ---
class MSAClassifier(nn.Module):
    def __init__(self, num_layers=4, dim=128, num_classes=2):
        super().__init__()
        self.encoder = MSAEncoder(num_layers=num_layers, dim=dim)

        self.classifier = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):  # x: (B, L, D)
        x = self.encoder(x)         # (B, L, D, C)
        x = x.mean(dim=2)           # mean over D → (B, L, C)
        x = x.mean(dim=1)           # mean over L → (B, C)
        out = self.classifier(x)    # (B, num_classes)
        return out


In [6]:
model

MSAClassifier(
  (encoder): MSAEncoder(
    (embeddings): MSAInputEmbedding(
      (embedding): Embedding(21, 128)
    )
    (blocks): ModuleList(
      (0-3): 4 x CrossAxialMambaMSA(
        (norm_L): MambaRMSNorm()
        (norm_D): MambaRMSNorm()
        (mamba_L): Mamba(
          (in_proj): Linear(in_features=128, out_features=256, bias=False)
          (conv1d): Conv1d(128, 128, kernel_size=(4,), stride=(1,), padding=(3,), groups=128)
          (act): SiLU()
          (x_proj): Linear(in_features=128, out_features=40, bias=False)
          (dt_proj): Linear(in_features=8, out_features=128, bias=True)
          (out_proj): Linear(in_features=128, out_features=128, bias=False)
        )
        (mamba_D): Mamba(
          (in_proj): Linear(in_features=128, out_features=256, bias=False)
          (conv1d): Conv1d(128, 128, kernel_size=(4,), stride=(1,), padding=(3,), groups=128)
          (act): SiLU()
          (x_proj): Linear(in_features=128, out_features=40, bias=False)
        

In [5]:
from torchinfo import summary
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MSAClassifier(num_layers=4, dim=128, num_classes=2).to(device)

dummy_input = torch.randint(low=0, high=21, size=(1, 61, 80)).long().to(device)

summary(
    model,
    input_data=(dummy_input,),
    col_names=["input_size", "output_size", "num_params"],
    row_settings=["var_names"]
)


Layer (type (var_name))                       Input Shape               Output Shape              Param #
MSAClassifier (MSAClassifier)                 [1, 61, 80]               [1, 2]                    --
├─MSAEncoder (encoder)                        [1, 61, 80]               [1, 61, 80, 128]          --
│    └─MSAInputEmbedding (embeddings)         [1, 61, 80]               [1, 61, 80, 128]          --
│    │    └─Embedding (embedding)             [1, 61, 80]               [1, 61, 80, 128]          2,688
│    └─ModuleList (blocks)                    --                        --                        --
│    │    └─CrossAxialMambaMSA (0)            [1, 61, 80, 128]          [1, 61, 80, 128]          116,736
│    │    └─CrossAxialMambaMSA (1)            [1, 61, 80, 128]          [1, 61, 80, 128]          116,736
│    │    └─CrossAxialMambaMSA (2)            [1, 61, 80, 128]          [1, 61, 80, 128]          116,736
│    │    └─CrossAxialMambaMSA (3)            [1, 61, 80, 128]      

In [5]:
# res_model = timm.create_model("hf_hub:timm/resnet50.a1_in1k", pretrained=True).to(device)
# summary(res_model, input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"], row_settings=["var_names"])

In [6]:
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import KFold

df = pd.read_csv("/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/rhapsody2_sav_db_exactmatch_only.tsv", sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 10-fold 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
splits = list(kf.split(df))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# oversampling: label == 1
pos_df = train_df[train_df["Label"] == 1]
neg_df = train_df[train_df["Label"] == 0]

repeat_factor = max(1, len(neg_df) // max(len(pos_df), 1))
oversampled_train_df = pd.concat([neg_df, pd.concat([pos_df] * repeat_factor)], ignore_index=True)
oversampled_train_df = oversampled_train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # 셔플

# Dataset
train_dataset = MSADataset(oversampled_train_df, "/mnt/e/CAGI_data/msa_dict_valid.pkl")
val_dataset   = MSADataset(val_df, "/mnt/e/CAGI_data/msa_dict_valid.pkl")

# 5. Dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

In [7]:
print("=== 오버샘플링 전 ===")
print(f"  원본 train_df: {len(train_df)}")
print(f"    - Label 0 개수: {len(neg_df)}")
print(f"    - Label 1 개수: {len(pos_df)}")

print("\n=== 오버샘플링 후 ===")
print(f"  oversampled_train_df: {len(oversampled_train_df)}")
print(f"    - Label 0 개수: {(oversampled_train_df['Label'] == 0).sum()}")
print(f"    - Label 1 개수: {(oversampled_train_df['Label'] == 1).sum()}")

print(f"  원본 val_df: {len(val_df)}")


=== 오버샘플링 전 ===
  원본 train_df: 89869
    - Label 0 개수: 58355
    - Label 1 개수: 31514

=== 오버샘플링 후 ===
  oversampled_train_df: 89869
    - Label 0 개수: 58355
    - Label 1 개수: 31514
  원본 val_df: 9986


In [8]:
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MSAClassifier(num_layers=6, dim=128).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

num_epochs = 100
best_pr_auc = 0.0
save_path = "/mnt/e/CAGI_data/best_model.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        x = batch["msa"].to(device)         # [B, L, D]
        y = batch["label"].to(device)       # [B]

        optimizer.zero_grad()
        logits = model(x)                   # [B, 2]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            x = batch["msa"].to(device)
            y = batch["label"].to(device)

            logits = model(x)
            loss = criterion(logits, y)

            probs = torch.softmax(logits, dim=1)[:, 1]  # P(class=1)

            val_loss += loss.item() * x.size(0)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader.dataset)
    pr_auc = average_precision_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PR-AUC: {pr_auc:.4f}")

    # --- Save best model ---
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), save_path)
        print(f">>> Best model saved! PR-AUC: {pr_auc:.4f}")


Epoch 1 [Val]: 100%|██████████| 313/313 [00:20<00:00, 15.47it/s]



Epoch 1/100
Train Loss: 0.5258 | Val Loss: 0.5015 | Val PR-AUC: 0.6779
>>> Best model saved! PR-AUC: 0.6779


Epoch 2 [Val]: 100%|██████████| 313/313 [00:19<00:00, 15.74it/s]



Epoch 2/100
Train Loss: 0.4732 | Val Loss: 0.4636 | Val PR-AUC: 0.7310
>>> Best model saved! PR-AUC: 0.7310


Epoch 3 [Val]: 100%|██████████| 313/313 [00:19<00:00, 16.11it/s]


...


Epoch 77/100
Train Loss: 0.0043 | Val Loss: 1.3731 | Val PR-AUC: 0.8328


Epoch 78 [Val]: 100%|██████████| 313/313 [00:19<00:00, 15.68it/s]



Epoch 78/100
Train Loss: 0.0042 | Val Loss: 1.4079 | Val PR-AUC: 0.8334


Epoch 79 [Train]:  57%|█████▋    | 1612/2809 [06:29<04:49,  4.14it/s]


KeyboardInterrupt: 